<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.1-burgers-2d/Ex09.1_00_setup_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.1 · Notebook 00 — setup and the reference solution

**Paired with L9.1 · Laminar Flow**

**Read and run; you are not asked to rewrite this.**

$$u_t + u\,u_x + v\,u_y = \nu\left(u_{xx}+u_{yy}\right), \qquad
v_t + u\,v_x + v\,v_y = \nu\left(v_{xx}+v_{yy}\right)$$

Two coupled, nonlinear equations — Navier–Stokes with the pressure and the
incompressibility constraint removed. Exact solution:

$$u = \tfrac34 - \frac{1}{4\left(1+e^{(-4x+4y-t)/32\nu}\right)}, \qquad
v = \tfrac34 + \frac{1}{4\left(1+e^{(-4x+4y-t)/32\nu}\right)}$$

With $U = L = 1$ the Reynolds number is simply $1/\nu$, so the control panel
exposes Re directly.

Setup instructions are in `README.md`.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-burgers-2d/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The problem, in numbers

In [ ]:
pb.describe_problem()

**What you should see.** The unit square, a time window of one, and
$\nu = 0.05$ against $\mathrm{Re} = 20$ — the default the rest of the set
starts from.

The line worth reading twice is the front width. The exponent of the exact
solution can be written

$$\frac{-4x + 4y - t}{32\nu} = \frac{(y - x) - t/4}{8\nu},$$

so the front sits on $y = x + t/4$ and its e-folding width in $y-x$ is
$8\nu$. Viscosity does not merely damp this solution, it **sets the length
scale the network has to resolve**, and the Reynolds number is the reciprocal
of it. That is the entire content of notebook 03, stated here in one line
before any training has happened.

---

## 2 · Verify the reference solution with autograd

Before trusting a reference, differentiate it. Both residuals should be zero
to machine precision, because the solution is exact and `grad` is exact.

Note the wrapping: `sample_domain` returns **NumPy**, like every sampler in
this course, and `to_tensor(..., requires_grad=True)` is what makes it
something autograd can differentiate.

In [ ]:
nu = 0.05
print("Re =", pb.reynolds_from_nu(nu))

xyt = to_tensor(pb.sample_domain(400), requires_grad=True)
x, y, t = xyt[:, 0:1], xyt[:, 1:2], xyt[:, 2:3]
E = torch.exp((-4*x + 4*y - t) / (32*nu))
u = 0.75 - 0.25/(1 + E)
v = 0.75 + 0.25/(1 + E)

gu, gv = grad(u, xyt), grad(v, xyt)
ru = gu[:,2:3] + u*gu[:,0:1] + v*gu[:,1:2] - nu*(d2(u,xyt,0)+d2(u,xyt,1))
rv = gv[:,2:3] + u*gv[:,0:1] + v*gv[:,1:2] - nu*(d2(v,xyt,0)+d2(v,xyt,1))
print(f"max |u residual| = {ru.abs().max().item():.3e}")
print(f"max |v residual| = {rv.abs().max().item():.3e}")
assert max(ru.abs().max().item(), rv.abs().max().item()) < 1e-6

**What you should see.** `Re = 20.0`, and two residuals comfortably
under the $10^{-6}$ the assertion demands.

Column 2 is the time derivative, columns 0 and 1 the spatial ones — space
first, time last, everywhere in this course. Getting that wrong is a silent
error: the code runs and solves a different equation.

---

## 3 · The three point sets

Three sets of points, three different jobs, and all three arrive as NumPy.

In [ ]:
xyt_f = pb.sample_domain(3000, seed=1)
xyt_b = pb.sample_boundary_xt(20, 12, seed=1)

print("collocation (PDE)   ", type(xyt_f).__name__, xyt_f.shape)
print("boundary + IC       ", type(xyt_b).__name__, xyt_b.shape)
print("distinct instants on the boundary tube:", len(np.unique(xyt_b[:, 2])))
print("points at t = 0                       :", int((xyt_b[:, 2] == 0).sum()))

fig = plt.figure(figsize=(7.6, 5.2))
ax = fig.add_subplot(projection="3d")
ax.scatter(xyt_f[::4, 0], xyt_f[::4, 1], xyt_f[::4, 2],
           s=2, alpha=0.35, color="#1f77b4", label="collocation")
ax.scatter(xyt_b[:, 0], xyt_b[:, 1], xyt_b[:, 2],
           s=4, alpha=0.7, color="#d94f2b", label="boundary + t = 0")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("t")
ax.set_title("The space–time slab, and what is sampled where")
ax.legend(fontsize=8, loc="upper left")
plt.show()

**What you should see.** A box of blue points, with orange points on
its four vertical faces and a sheet of them across its base.

The base is the initial condition and the faces are the boundary condition.
Both are taken from the exact solution, which makes this a **verification**
problem: any error the later notebooks measure belongs to the model, never to
the data.

---

## 4 · What the solution looks like

In [ ]:
g = np.linspace(0, 1, 120); X, Y = np.meshgrid(g, g)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, t in zip(ax, [0.0, 0.5, 1.0]):
    im = a.contourf(X, Y, pb.exact_u(X, Y, t, nu), 40, cmap="magma")
    a.set_title(f"exact u,  t = {t}"); a.set_aspect("equal"); fig.colorbar(im, ax=a)
plt.tight_layout(); plt.show()

A front travelling along the diagonal. Lower $\nu$ (higher Re) makes it
steeper — which is exactly what will make training harder in notebook 03.

In [ ]:
plt.figure(figsize=(6.5, 3.2))
for Re in (10, 40, 200):
    s = np.linspace(-0.5, 0.5, 400)
    plt.plot(s, pb.exact_u(s, -s, 0.0, pb.nu_from_reynolds(Re)), label=f"Re = {Re}")
plt.xlabel("distance across the front"); plt.ylabel("u"); plt.legend()
plt.tight_layout(); plt.show()

---

## 5 · Ready

Next: **notebook 01**, where you write the two residuals and the loss. That is
the only notebook in this set with anything to write; everything afterwards
calls what you produce there.